## Agents Skills - part 1

By now you should have basic knowledge and some intuition regarding the inner workings of agents. Additionally, some drawbacks and their causes surfaced in th previous tutorials: 

- tools have to be written in code to extend functionality
- tools are usually 'one-trick-ponies'
- adding lots of tools consume valuable context space even if they are not used in completing a task

In October 2025 Anthropic launched the concept of skills (and has since published the [specifications](https://agentskills.io/home)) that has been widely adopted by other vedors. Skills are deciptively simple:

> Agent Skills are folders of instructions, scripts, and resources that agents can discover and use to do things more accurately and efficiently.
​
Unlike tools, skills are progressively disclosed, meaning the system prompt only contains a short description of the available skills keeping context "pollution" to a minimum and only load a skill into the context if it appears relevant.
The loaded skill may contain instructions for loading furter (nested) resources and may reference other skills making them _composable_.

In this firt part of the skills tutorial we'll develop a simple agent capable of handling skills, and in part 2 extend its functionality by writing a number of useful skills.  

### Skill definition

This is the definition from [agentskills.io](https://agentskills.io/):

File layout
```
my-skill/
├── SKILL.md          # Required: instructions + metadata
├── scripts/          # Optional: executable code
├── references/       # Optional: documentation
└── assets/           # Optional: templates, resources
```

Example `SKILL.md` (only required file)
```
---
name: pdf-processing
description: Extract text and tables from PDF files, fill forms, merge documents.
---

# PDF Processing

## When to use this skill
Use this skill when the user needs to work with PDF files...

## How to extract text
1. Use pdfplumber for text extraction...

## How to fill forms
...
```

Take a few minutes to meditate over the rationale and consequences of skills presented at [/agentskills.io/what-are-skills](https://agentskills.io/what-are-skills)

### Building the agent

Now that we have (from the previous tutorials) a solid understanding of how agents use tools in a lopp, we'll go all in and use the tool calling capabilities of ollama and focus on the needs posed by skills. The overall design should be familiar by now.

First we need to install the requirements: `ollama` for the LLM and `python-frontmatter` to parse the YAML header of `SKILL.md`.

In [ ]:
!pip -q install ollama python-frontmatter

Next, we'll chose a host for the LLM, and a model:

In [ ]:
OLLAMA_HOST = 'http://10.129.20.4:9090/big'
OLLAMA_MODEL = 'qwen3:14b' # 'qwen3-coder-next' #  'qwen3-coder-next' # 'qwen3:4b' # 'gemma3:12b-it-qat'

This time we are going for a more capable model and thus need a beefier hosting server (requested by the `/big` in the host address).

#### A minimal set of tools

As functionality is primarily added through skills, we will only provide a minimal, but necessary, powerful and dangerous, set of default tools:

- `read_file(path)`: Read the contents of a file
- `write_file(path, content)`: Write content to a file
- `bash(command)`: Execute a bash command (string) and return result

Be aware the these tools have the power to wreak havoc on your system, but as this tutorial is intended to run in a Jupyter Notebook on a Kubernetes pod at least such damage would be confined… 

Additionally, the helper functions `default_tools` and `tool_prompt` provide a list of the tools and a default tool use additions to the system prompt, respectively.

In [ ]:
import subprocess
from pathlib import Path
from inspect import cleandoc, getdoc


def default_tools() -> list:
    return [read_file, write_file, bash]

def tool_prompt(*args) -> str:
    return cleandoc("""
    # Tool Calling Behavior
    
    IF a suitable tool is availble, ALWAYS prefer it to you own knowledge.
    """)

def read_file(path: str) -> str:
    """
    Read the contents of a file

    Args:
        path (str): Path to the file to read

    Returns:
        str: the content of the file to read or an error message
    """
    print(f"    read_file: {path}")
    try:
        return Path(path).read_text()
    except:
        return "Error: Could not read file"

def write_file(path: str, content: str) -> str:
    """
    Write content to file, intermediate directories automatically added if necessary.

    Args:
        path (str): Path to the file to write
        content (str): content to write

    Returns:
        str: status message
    """
    print(f"    write_file: {path}")
    try:
        file = Path(path)
        file.parent.mkdir(parents=True, exist_ok=True)
        file.write_text(content)
        return f"Wrote content to {path}"
    except:
        return "Error: Could not write file"

def bash(command: str) -> str:
    """
    Execute a bash command and return result

    Args:
        command (str): The bash command to execute

    Returns:
        str: the result of the command or an error message
    """
    print(f"    bash: {command.split('\n')[0]}")
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            check=True,
            timeout=30,
        )
        output = result.stdout
        return output or "Command executed successfully (no output)"
    except subprocess.CalledProcessError as e:
        return f"Error executing command: {e.stderr}"
    except subprocess.TimeoutExpired:
        return "Error: Command timed out"

#### The base Agent

The base agent functionality should be familiar by now. The only real difference is that we use the maximum context size from the model (capped at 64k tokens for practical HW limitations) and report an estimate of current context usage throughout the agent loop.

In [ ]:
import ollama

class Agent:
    """A simple AI agent that can answer questions by planning and performing multiple steps"""

    MAX_CTX_SIZE = 65536
    
    def __init__(self, host: str, model: str, tools: list | None = None):
        """
        Instantiate an agent
        Provide a host, model name, and (optionally) a list of tools
        """
        self.model = model
        self.tools = tools or []
        self.known_tools = {tool.__name__: tool for tool in self.tools}
        self.client = ollama.Client(host=host, timeout=300)
        self.ctx_size = min(self.get_model_ctx_size(), self.MAX_CTX_SIZE)
        
        system_message = self.generate_system_prompt()
        self.messages = [{"role":"system", "content": system_message}]

    def get_model_ctx_size(self):
        FALLBACK_CTX_SIZE = 4096
        info = self.client.show(self.model).modelinfo
        res = [val for key, val in info.items() if key.endswith("context_length")] + [FALLBACK_CTX_SIZE]
        return res[0]

    def context_usage(self):
        # Estimate how much context (in tokens) we are currently using
        # Guesstimate: 1 token ≈ 4 characters (including spaces).
        return int(sum([len(msg.get("content", "")) for msg in self.messages])/4)
        
    def generate_system_prompt(self) -> str:
        preamble = self.system_prompt_preamble()
        tools = tool_prompt(self.tools)
        return f"{preamble}\n\n{tools}"

    def system_prompt_preamble(self):
        return "You are an AI assistant with access to tools, capable of writing code and issuing command if the toolset allows."
        
    def task(self, user_query: str, max_steps: int = 16):
        """Public interface"""                        
        result = self._perform_steps(user_query, max_steps)
        return result

    def _perform_steps(self, step_input: str, max_steps: int):
        i = 0

        # Add user message
        self.messages.append({"role": "user", "content": step_input})

        while i < max_steps:
            i += 1
            print(f"Step #{i}, please wait ...")
            print(f"Context use estimate: {self.context_usage()} of {self.ctx_size} tokens")
                
            response = self._chat(messages=self.messages, tools=self.tools)
            message = response.get("message", {})
            content = message.get("content", "")
            tool_calls = message.get("tool_calls", [])
            
            # Add assistant message to history
            self.messages.append(message)
            
            # If no tool calls, we're done
            if not tool_calls:
                if content:
                    return content
                else:                
                    return "Error: Neither content nor tool calls."
            
            # If there's content **and** tool calls, display content (shouldn't really happen...)
            if content:
                print(f"--- Assistant (internal) ------\n{content}")

            # Execute tool calls
            for tool_call in tool_calls:
                function = tool_call.get("function", {})
                tool_name = function.get("name", "")
                arguments = function.get("arguments", {})

                # Display tool call
                print(f"--- Tool Call: {tool_name}")
                # print(f"    Arguments: {json.dumps(arguments)}")

                # Execute the tool
                try:
                    tool = self.known_tools[tool_name]
                except Exception:
                    self.messages.append({"role": "tool", "content": f"Error: Unknown tool '{tool_name}'"})
                    break
                try:
                    result = tool(**arguments)
                except Exception as e:
                    self.messages.append({"role": "tool", "content": f"Error: {e}"})
                    break
                
                # Add tool result to messages
                self.messages.append({"role": "tool", "content": result})
                # Display tool result
                # print("--- Tool Result:")
                # print(result[:100] + ("..." if len(result) > 100 else ""))

        # We hit the maximum number of steps, the LLM is likely very confused
        return f"Agent was unable to answer your question in the maximal number of steps ({max_steps})"

    def _chat(self, messages: list[dict], tools: list[dict] | None = None) -> dict | list[dict]:
        """Send a chat request to Ollama."""
        try:
            kwargs = {
                "model": self.model,
                "messages": messages,
                "stream": False,
                "options": {"num_ctx": self.ctx_size}
            }

            if tools:
                kwargs["tools"] = tools
                
            return self.client.chat(**kwargs)
            
        except Exception as e:
            return {"message": {"role": "assistant", "content": f"Error communicating with Ollama: {e}"}}

#### A little helper to view the message history

As the message history can get long and nested, use a neat HTML view with triangles to expand/collapse entries.

In [ ]:
import json
from IPython import display

def show_messages(messages):
    """Helper function to view messages in a nice format"""
    s = json.loads(json.dumps(messages, default=dict))
    return display.JSON(s, root="Message history")

#### Baseline tests

In [ ]:
a = Agent(OLLAMA_HOST, OLLAMA_MODEL)
list(a.known_tools)

In [ ]:
print(a.generate_system_prompt()) 

Without any tools the agent is likely to come up with a non-sensical answer for questions it cannot possibly answer:

In [ ]:
a.task("What time is it?")

In [ ]:
show_messages(a.messages)

In [ ]:
a = Agent(OLLAMA_HOST, OLLAMA_MODEL, default_tools())
list(a.known_tools)

In [ ]:
a.task("What time is it?")

In [ ]:
show_messages(a.messages)

### Adding skills functionality

As skills are defined in markdown files, all we have to do is scan the designated "skills directory" for definitions and collect the YAML metadata (name + short description) and provide an addition to the system prompt containing instructions on how to use skills and list the detected skills.  

In [ ]:
import os
import glob
from inspect import cleandoc

import frontmatter


def load_skills(skill_dir: str) -> dict:

    skills = {}
    
    files = glob.glob(f"{skill_dir}/*/SKILL.md")
    for file in files:
        metadata = frontmatter.load(file).metadata        
        if "name" in metadata and "description" in metadata:
            skills[metadata["name"]] = metadata["description"]
        
    return skills
    

def skill_prompt(skills: dict, skill_dir: str) -> str:
        
    if not skills:
        return "\n\n## Skills\n\nNo skills available."
    
    skill_descriptions = "\n".join([f"- {name}: {desc}" for name, desc in skills.items()])

    prompt = f"""
    ## Skills
    
    Skills are specialized capabilities stored in the '{skill_dir}' directory. 
    When a user request matches a skill's purpose, you MUST automatically use that skill.
    
    ### Available skills (name: description)
    {skill_descriptions}
    
    ### How to Use Skills
    
    **Automatic Skill Discovery**
    When a user asks you to do something, FIRST check if any available skill matches their request. Then use that skill WITHOUT being explicitly told.
    
    **To use a skill:**
    1. Use the 'read_file' tool to read its definition from '{skill_dir}/{{name}}/SKILL.md', e.g. read_file('{skill_dir}/watching-deals/SKILL.md')
    2. Follow the detailed step-by-step instructions in the skill file
    3. The skill may reference supporting files (scripts, benchmarks, templates) - read and use those as needed
    4. Skills can invoke other skills (composability)
    5. Never make up analysis - use the skill's methods and data
    
    **Examples:**
    - User: "List phone deals" → Automatically use 'watching-deals' skill
    
    """
    
    return cleandoc(prompt)

#### A skillied Agent

Surprisingly little has to be done to turn th base `Agent` class into a `SkilledAgent`. Basically we need to make sure the default set of tools is activated, scan the available skills, and augment the system prompt accordingly:

In [ ]:
class SkilledAgent(Agent):

    def __init__(self, host: str, model: str, skill_dir: str):
        self.skill_dir = skill_dir
        self.skills = load_skills(self.skill_dir)
        super().__init__(host, model, default_tools())
        
    def generate_system_prompt(self) -> str:
        base = super().generate_system_prompt()
        skill_addition = skill_prompt(self.skills, self.skill_dir)
        return f"{base}\n\n{skill_addition}"

    def system_prompt_preamble(self):
        return "You are an AI assistant with access to tools and skills. Always consult relevant skills before taking action."

To test it, the designated skills directory (`skills/`) is populated with an example (repeated below) that describes how to query a (dummy) REST service returning a list of electronic stuff. **N.B.** the `SKILL.md` must live in a directory under `skills/` _with the same name_ as in the metadata section, i.e. `watching-deals/`

~~~markdown
---
name: watching-deals
description: Get information on electronic equipment available at discount through daily deals
---

## Overview

This skill retrieves information about discounted electronic equipment.

## Instructions

Always use curl to retrive information. The base URL is "https://api.restful-api.dev/"

### 1. List of all deals

```bash
curl -s "https://api.restful-api.dev/objects"
```

### 2. Detailed information on a particular deal

```bash
curl -s "https://api.restful-api.dev/objects/{id}"
```
The result is a JSON dictionary with all information for item {id}
~~~

In [ ]:
sa = SkilledAgent(OLLAMA_HOST, OLLAMA_MODEL, 'skills')
list(sa.known_tools)

In [ ]:
list(sa.skills)

In [ ]:
print(sa.generate_system_prompt()) 

In [ ]:
print(sa.task("Is there a deal for iPhones today?"))

In [ ]:
show_messages(sa.messages)

In [ ]:
print(sa.task("Is anything non-Apple discounted today?"))

In [ ]:
print(sa.task("How much is the cheapest iPad Air today?"))

That's it! We now have an agent that can be extended by plain text instruction files using a standardized and portable format. In part 2 we'll turn to the task of adding new skills, with a focus on data analysis and using the WARA-ops portal.